# Reproduction of the manuscript's quantitative analyses

This notebook contains the calculations for the manuscript's two principal statistical results. The first analysis uses 10,000 seeded permutations to compare repeated candidate identification with the frequency expected after preserving the accessions or lines assessed and number of candidates in each independent screen. The second obtains the exact distribution of the combined multidisease count from hypergeometric probability masses and discrete convolution.

The notebook writes four tables containing these calculated results. Run it from `reproducibility/code`; it reads the supplied data from this package.


In [1]:
from math import comb
from pathlib import Path
from typing import Sequence
import os

import numpy as np
import pandas as pd
from IPython.display import display

PERMUTATION_N = 10_000
PERMUTATION_SEED = 20_260_714
CHUNK_SIZE = 500

CODE_DIR = Path.cwd().resolve()
PACKAGE_ROOT = Path(os.environ.get("ATLAS_PACKAGE_ROOT", CODE_DIR.parents[1])).resolve()
INPUT_DIR = PACKAGE_ROOT / "reproducibility" / "data" / "analysis_inputs"
OUTPUT_DIR = Path(
    os.environ.get("ATLAS_OUTPUT_DIR", CODE_DIR / "notebook_outputs")
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

recurrence_input = INPUT_DIR / "Recurrence_Analysis_Matrix_Anonymous.csv"
fixed_margin_input = INPUT_DIR / "Fixed_Margin_Comparison_Blocks.csv"


## Repeated candidate identification

The analysis includes each anonymous accession or line in every independent screen where it was assessed. Within every permutation, the calculation preserves the accessions or lines assessed and the observed number meeting the documented criterion in each screen, then randomly selects the same number of candidates without replacement. The statistic is the proportion identified in at least two screens among candidates identified at least once and assessed in at least two screens.


In [2]:
required_recurrence_columns = {
    "analysis_identity_id",
    "evidence_set_id",
    "candidate_selected",
}

design = pd.read_csv(
    recurrence_input,
    dtype={"analysis_identity_id": str, "evidence_set_id": str},
)
missing = sorted(required_recurrence_columns - set(design.columns))
if missing:
    raise ValueError(f"Recurrence input is missing columns: {missing}")
design = design.loc[:, sorted(required_recurrence_columns)].copy()
if design[["analysis_identity_id", "evidence_set_id"]].isna().any().any():
    raise ValueError("An accession or line identifier or independent screen identifier is blank")
design["candidate_selected"] = pd.to_numeric(
    design["candidate_selected"], errors="raise"
).astype(np.int8)
if not design["candidate_selected"].isin([0, 1]).all():
    raise ValueError("Candidate classifications must be 0 or 1")
if design.duplicated(["analysis_identity_id", "evidence_set_id"]).any():
    raise ValueError("Duplicate combination of accession or line and independent screen")
design = design.sort_values(
    ["analysis_identity_id", "evidence_set_id"]
).reset_index(drop=True)

display(
    pd.DataFrame(
        {
            "Quantity": ["Accession or line observations across independent screens", "Independent screens", "Anonymous accessions or lines"],
            "Count": [
                len(design),
                design["evidence_set_id"].nunique(),
                design["analysis_identity_id"].nunique(),
            ],
        }
    )
)


,Quantity,Count
0,Accession or line observations across independ...,5163
1,Independent screens,34
2,Anonymous accessions or lines,3655


In [3]:
def recurrence_expectations(design: pd.DataFrame) -> dict[str, float | int]:
    set_rates = design.groupby("evidence_set_id").agg(
        assessed_identity_n=("analysis_identity_id", "nunique"),
        candidate_n=("candidate_selected", "sum"),
    )
    set_rates["candidate_rate"] = (
        set_rates["candidate_n"] / set_rates["assessed_identity_n"]
    )

    by_identity = design.groupby("analysis_identity_id").agg(
        opportunity_n=("evidence_set_id", "nunique"),
        selected_set_n=("candidate_selected", "sum"),
    )
    eligible = by_identity.index[by_identity["opportunity_n"].ge(2)]
    eligible_counts = by_identity.loc[eligible, "selected_set_n"]
    observed_once = int(eligible_counts.eq(1).sum())
    observed_repeated = int(eligible_counts.ge(2).sum())

    expected_once = 0.0
    expected_repeated = 0.0
    expected_any = 0.0
    for identity in eligible:
        sets_for_identity = design.loc[
            design["analysis_identity_id"].eq(identity), "evidence_set_id"
        ]
        probabilities = [
            float(set_rates.loc[set_id, "candidate_rate"])
            for set_id in sets_for_identity
        ]
        probability_mass = [1.0]
        for probability in probabilities:
            updated = [0.0] * (len(probability_mass) + 1)
            for selected_n, mass in enumerate(probability_mass):
                updated[selected_n] += mass * (1.0 - probability)
                updated[selected_n + 1] += mass * probability
            probability_mass = updated
        expected_once += probability_mass[1]
        expected_repeated += sum(probability_mass[2:])
        expected_any += sum(probability_mass[1:])

    observed_any = observed_once + observed_repeated
    return {
        "evidence_set_n": int(design["evidence_set_id"].nunique()),
        "anonymous_identity_n": int(design["analysis_identity_id"].nunique()),
        "identity_n_two_plus_opportunities": int(len(eligible)),
        "candidate_single_opportunity_n": int(
            (
                by_identity["opportunity_n"].eq(1)
                & by_identity["selected_set_n"].gt(0)
            ).sum()
        ),
        "observed_candidate_n": observed_any,
        "observed_identified_once_n": observed_once,
        "observed_identified_repeatedly_n": observed_repeated,
        "observed_identified_repeatedly_fraction": observed_repeated / observed_any,
        "expected_candidate_n": expected_any,
        "expected_identified_once_n": expected_once,
        "expected_identified_repeatedly_n": expected_repeated,
        "expected_identified_repeatedly_fraction": expected_repeated / expected_any,
    }


recurrence_observed = recurrence_expectations(design)
observed_overview = pd.DataFrame(
    {
        "Quantity": [
            "Independent screens",
            "Anonymous accessions or lines",
            "Accessions or lines assessed in at least two screens",
            "Candidates assessed in one independent screen only",
            "Candidates assessed in at least two screens",
            "Identified in one screen",
            "Identified in at least two screens",
        ],
        "Value": [
            recurrence_observed["evidence_set_n"],
            recurrence_observed["anonymous_identity_n"],
            recurrence_observed["identity_n_two_plus_opportunities"],
            recurrence_observed["candidate_single_opportunity_n"],
            recurrence_observed["observed_candidate_n"],
            recurrence_observed["observed_identified_once_n"],
            recurrence_observed["observed_identified_repeatedly_n"],
        ],
    }
)
display(observed_overview)


,Quantity,Value
0,Independent screens,34
1,Anonymous accessions or lines,3655
2,Accessions or lines assessed in at least two s...,1154
3,Candidates assessed in one independent screen ...,1335
4,Candidates assessed in at least two screens,612
5,Identified in one screen,441
6,Identified in at least two screens,171


In [4]:
def recurrence_permutations(
    design: pd.DataFrame,
    permutation_n: int = PERMUTATION_N,
    seed: int = PERMUTATION_SEED,
    chunk_size: int = CHUNK_SIZE,
) -> pd.DataFrame:
    identities = sorted(set(design["analysis_identity_id"]))
    identity_index = {identity: index for index, identity in enumerate(identities)}
    opportunity = design.groupby("analysis_identity_id").size()
    included_in_statistic = np.array(
        [opportunity[identity] >= 2 for identity in identities]
    )

    set_designs: list[tuple[np.ndarray, int]] = []
    for _, group in design.groupby("evidence_set_id", sort=True):
        available = np.array(
            [identity_index[item] for item in group["analysis_identity_id"]],
            dtype=np.int32,
        )
        set_designs.append((available, int(group["candidate_selected"].sum())))

    rng = np.random.default_rng(seed)
    once_results: list[np.ndarray] = []
    repeated_results: list[np.ndarray] = []
    fraction_results: list[np.ndarray] = []
    for start in range(0, permutation_n, chunk_size):
        size = min(chunk_size, permutation_n - start)
        selection_counts = np.zeros((size, len(identities)), dtype=np.int16)
        row_indices = np.arange(size)[:, None]
        for available, selected_n in set_designs:
            if selected_n == 0:
                continue
            if selected_n == len(available):
                selection_counts[:, available] += 1
                continue
            random_values = rng.random((size, len(available)))
            chosen_columns = np.argpartition(
                random_values, selected_n - 1, axis=1
            )[:, :selected_n]
            chosen_identities = available[chosen_columns]
            np.add.at(selection_counts, (row_indices, chosen_identities), 1)

        eligible_counts = selection_counts[:, included_in_statistic]
        once = np.count_nonzero(eligible_counts == 1, axis=1)
        repeated = np.count_nonzero(eligible_counts >= 2, axis=1)
        selected_any = once + repeated
        once_results.append(once)
        repeated_results.append(repeated)
        fraction_results.append(
            np.divide(
                repeated,
                selected_any,
                out=np.full(size, np.nan, dtype=float),
                where=selected_any > 0,
            )
        )

    return pd.DataFrame(
        {
            "permutation": np.arange(1, permutation_n + 1),
            "identified_once_n": np.concatenate(once_results),
            "identified_repeatedly_n": np.concatenate(repeated_results),
            "identified_repeatedly_fraction": np.concatenate(fraction_results),
        }
    )


recurrence_distribution = recurrence_permutations(design)
display(
    recurrence_distribution.head().rename(
        columns={
            "permutation": "Permutation",
            "identified_once_n": "Identified once",
            "identified_repeatedly_n": "Identified repeatedly",
            "identified_repeatedly_fraction": "Proportion identified repeatedly",
        }
    )
)


,Permutation,Identified once,Identified repeatedly,Proportion identified repeatedly
0,1,509,139,0.214506
1,2,504,138,0.214953
2,3,522,131,0.200613
3,4,498,138,0.216981
4,5,495,138,0.218009


In [5]:
repeated_n = recurrence_distribution["identified_repeatedly_n"].to_numpy()
repeated_fraction = recurrence_distribution[
    "identified_repeatedly_fraction"
].to_numpy()
observed_n = int(recurrence_observed["observed_identified_repeatedly_n"])
observed_fraction = float(
    recurrence_observed["observed_identified_repeatedly_fraction"]
)
lower_tail = (
    int(np.count_nonzero(repeated_fraction <= observed_fraction)) + 1
) / (len(repeated_fraction) + 1)
upper_tail = (
    int(np.count_nonzero(repeated_fraction >= observed_fraction)) + 1
) / (len(repeated_fraction) + 1)

recurrence_summary = {
    **recurrence_observed,
    "permutation_n": len(recurrence_distribution),
    "permutation_seed": PERMUTATION_SEED,
    "permutation_identified_once_n_median": float(
        np.median(recurrence_distribution["identified_once_n"])
    ),
    "permutation_identified_repeatedly_n_median": float(np.median(repeated_n)),
    "permutation_identified_repeatedly_fraction_median": float(
        np.nanmedian(repeated_fraction)
    ),
    "permutation_identified_repeatedly_fraction_lower_95": float(
        np.nanquantile(repeated_fraction, 0.025)
    ),
    "permutation_identified_repeatedly_fraction_upper_95": float(
        np.nanquantile(repeated_fraction, 0.975)
    ),
    "empirical_p_identified_repeatedly_at_least_observed": (
        int(np.count_nonzero(repeated_n >= observed_n)) + 1
    )
    / (len(repeated_n) + 1),
    "empirical_two_sided_p_identified_repeatedly_fraction": min(
        1.0, 2.0 * min(lower_tail, upper_tail)
    ),
}

recurrence_output = pd.DataFrame(
    [
        {
            "Independent screens (n)": recurrence_summary["evidence_set_n"],
            "Anonymous accessions or lines (n)": recurrence_summary["anonymous_identity_n"],
            "Assessed in at least two screens (n)": recurrence_summary["identity_n_two_plus_opportunities"],
            "Candidates assessed in one independent screen only (n)": recurrence_summary["candidate_single_opportunity_n"],
            "Candidates assessed in at least two screens (n)": recurrence_summary["observed_candidate_n"],
            "Candidates identified once (n)": recurrence_summary["observed_identified_once_n"],
            "Candidates identified repeatedly (n)": recurrence_summary["observed_identified_repeatedly_n"],
            "Observed proportion identified repeatedly": recurrence_summary["observed_identified_repeatedly_fraction"],
            "Permutation median": recurrence_summary["permutation_identified_repeatedly_fraction_median"],
            "Central 95% permutation interval, lower limit": recurrence_summary["permutation_identified_repeatedly_fraction_lower_95"],
            "Central 95% permutation interval, upper limit": recurrence_summary["permutation_identified_repeatedly_fraction_upper_95"],
            "Two-sided empirical P": recurrence_summary["empirical_two_sided_p_identified_repeatedly_fraction"],
            "Permutations (n)": recurrence_summary["permutation_n"],
            "Random seed": recurrence_summary["permutation_seed"],
        }
    ]
)
recurrence_output.to_csv(
    OUTPUT_DIR / "Repeated_Identification_Summary.csv", index=False
)
recurrence_distribution.rename(
    columns={
        "permutation": "Permutation",
        "identified_once_n": "Candidates identified once (n)",
        "identified_repeatedly_n": "Candidates identified repeatedly (n)",
        "identified_repeatedly_fraction": "Proportion identified repeatedly",
    }
).to_csv(OUTPUT_DIR / "Repeated_Identification_Permutation_Distribution.csv", index=False)

recurrence_report = pd.DataFrame(
    {
        "Result": [
            "Candidates assessed in at least two independent screens",
            "Identified once",
            "Identified repeatedly",
            "Observed proportion",
            "Permutation median",
            "Central 95% interval",
            "Two-sided empirical P",
        ],
        "Value": [
            recurrence_summary["observed_candidate_n"],
            recurrence_summary["observed_identified_once_n"],
            recurrence_summary["observed_identified_repeatedly_n"],
            f'{100 * recurrence_summary["observed_identified_repeatedly_fraction"]:.1f}%',
            f'{100 * recurrence_summary["permutation_identified_repeatedly_fraction_median"]:.1f}%',
            (
                f'{100 * recurrence_summary["permutation_identified_repeatedly_fraction_lower_95"]:.1f}% to '
                f'{100 * recurrence_summary["permutation_identified_repeatedly_fraction_upper_95"]:.1f}%'
            ),
            f'{recurrence_summary["empirical_two_sided_p_identified_repeatedly_fraction"]:.4f}',
        ],
    }
)
display(recurrence_report)


,Result,Value
0,Candidates assessed in at least two independen...,612
1,Identified once,441
2,Identified repeatedly,171
3,Observed proportion,27.9%
4,Permutation median,21.5%
5,Central 95% interval,19.1% to 23.8%
6,Two-sided empirical P,0.0002


## Exact distribution for direct disease comparisons

For a two-disease comparison, the overlap follows a hypergeometric distribution after conditioning on the number of accessions or lines and the number classified favorably for each disease. For a three-disease comparison, the calculation sums over the possible overlap of the first two diseases and then over the ways in which the third disease intersects accessions or lines classified favorably for exactly one of the first two. Discrete convolution combines the five independent comparison distributions. This calculation is deterministic.


In [6]:
def hypergeometric_overlap_pmf(
    population_n: int, first_selected_n: int, second_selected_n: int
) -> dict[int, float]:
    lower = max(0, second_selected_n - (population_n - first_selected_n))
    upper = min(first_selected_n, second_selected_n)
    denominator = comb(population_n, second_selected_n)
    return {
        overlap_n: (
            comb(first_selected_n, overlap_n)
            * comb(
                population_n - first_selected_n,
                second_selected_n - overlap_n,
            )
            / denominator
        )
        for overlap_n in range(lower, upper + 1)
    }


def fixed_margin_multidisease_pmf(
    population_n: int, disease_candidate_counts: Sequence[int]
) -> dict[int, float]:
    counts = tuple(int(value) for value in disease_candidate_counts)
    if len(counts) == 2:
        return hypergeometric_overlap_pmf(population_n, counts[0], counts[1])
    if len(counts) != 3:
        raise ValueError("Each comparison must contain two or three disease counts")

    first_n, second_n, third_n = counts
    first_second = hypergeometric_overlap_pmf(population_n, first_n, second_n)
    result: dict[int, float] = {}
    for overlap_n, overlap_probability in first_second.items():
        exactly_one_of_first_two_n = first_n + second_n - 2 * overlap_n
        third_overlap = hypergeometric_overlap_pmf(
            population_n, exactly_one_of_first_two_n, third_n
        )
        for third_additional_n, third_probability in third_overlap.items():
            multidisease_n = overlap_n + third_additional_n
            result[multidisease_n] = result.get(multidisease_n, 0.0) + (
                overlap_probability * third_probability
            )
    return result


def convolve_pmfs(
    first: dict[int, float], second: dict[int, float]
) -> dict[int, float]:
    result: dict[int, float] = {}
    for first_value, first_probability in first.items():
        for second_value, second_probability in second.items():
            total = first_value + second_value
            result[total] = result.get(total, 0.0) + (
                first_probability * second_probability
            )
    return result


def pmf_quantile(pmf: dict[int, float], probability: float) -> int:
    cumulative = 0.0
    for value in sorted(pmf):
        cumulative += pmf[value]
        if cumulative + 1e-15 >= probability:
            return value
    return max(pmf)


def pmf_summary(
    pmf: dict[int, float], observed_n: int
) -> dict[str, float | int | str]:
    lower_tail = sum(
        probability for value, probability in pmf.items() if value <= observed_n
    )
    upper_tail = sum(
        probability for value, probability in pmf.items() if value >= observed_n
    )
    return {
        "observed_multidisease_n": observed_n,
        "exact_null_mean": sum(
            value * probability for value, probability in pmf.items()
        ),
        "null_median": pmf_quantile(pmf, 0.5),
        "null_lower_95": pmf_quantile(pmf, 0.025),
        "null_upper_95": pmf_quantile(pmf, 0.975),
        "lower_tail_probability": lower_tail,
        "upper_tail_probability": upper_tail,
    }


def parse_counts(value: object) -> tuple[int, ...]:
    parts = [item.strip() for item in str(value).split("|") if item.strip()]
    counts = tuple(int(item.rsplit("=", 1)[-1]) for item in parts)
    if len(counts) not in {2, 3}:
        raise ValueError(f"Expected two or three disease counts, found {value!r}")
    return counts


In [7]:
required_fixed_margin_columns = {
    "comparison_id",
    "comparison_label",
    "diseases",
    "complete_case_n",
    "disease_candidate_counts",
    "observed_multidisease_n",
}
blocks = pd.read_csv(fixed_margin_input)
missing = sorted(required_fixed_margin_columns - set(blocks.columns))
if missing:
    raise ValueError(f"Direct-comparison input is missing columns: {missing}")
if len(blocks) != 5:
    raise ValueError(f"Expected five direct comparisons, found {len(blocks)}")
if blocks["comparison_id"].duplicated().any():
    raise ValueError("Comparison identifiers must be unique")

combined_pmf = {0: 1.0}
combined_observed_n = 0
combined_complete_case_n = 0
block_summaries: list[dict[str, object]] = []

for row in blocks.itertuples(index=False):
    population_n = int(row.complete_case_n)
    counts = parse_counts(row.disease_candidate_counts)
    if any(count < 0 or count > population_n for count in counts):
        raise ValueError(f"Candidate count outside 0 to N for {row.comparison_id}")
    observed_n = int(row.observed_multidisease_n)
    disease_names = [name.strip() for name in row.diseases.split(";")]
    if len(disease_names) != len(counts):
        raise ValueError(f"Disease labels and candidate counts disagree for {row.comparison_id}")
    disease_counts_text = "; ".join(
        f"{name}: {count}" for name, count in zip(disease_names, counts, strict=True)
    )
    pmf = fixed_margin_multidisease_pmf(population_n, counts)
    summary = pmf_summary(pmf, observed_n)
    block_summaries.append(
        {
            "comparison_id": row.comparison_id,
            "comparison_label": row.comparison_label,
            "diseases": row.diseases,
            "complete_case_n": population_n,
            "disease_candidate_counts": disease_counts_text,
            **summary,
        }
    )
    combined_pmf = convolve_pmfs(combined_pmf, pmf)
    combined_observed_n += observed_n
    combined_complete_case_n += population_n

combined_summary = {
    "comparison_id": "declared_multidisease_comparisons_fixed_margin",
    "comparison_label": "Sum across the five comparisons",
    "diseases": "",
    "complete_case_n": combined_complete_case_n,
    "disease_candidate_counts": "",
    **pmf_summary(combined_pmf, combined_observed_n),
}
block_summaries.append(combined_summary)

disease_overlap_output = pd.DataFrame(block_summaries).rename(
    columns={
        "comparison_id": "Comparison identifier",
        "comparison_label": "Comparison",
        "diseases": "Diseases",
        "complete_case_n": "Accessions or lines assessed (n)",
        "disease_candidate_counts": "Candidates for each disease (n)",
        "observed_multidisease_n": "Observed multidisease classifications (n)",
        "exact_null_mean": "Exact expected mean",
        "null_median": "Expected median",
        "null_lower_95": "Central 95% interval, lower limit",
        "null_upper_95": "Central 95% interval, upper limit",
        "lower_tail_probability": "Lower-tail probability",
        "upper_tail_probability": "Upper-tail probability",
    }
)
disease_overlap_output.to_csv(
    OUTPUT_DIR / "Disease_Overlap_Summary.csv", index=False
)
fixed_margin_distribution = pd.DataFrame(
    {
        "Multidisease classifications (n)": sorted(combined_pmf),
        "Exact probability": [combined_pmf[value] for value in sorted(combined_pmf)],
    }
)
fixed_margin_distribution.to_csv(
    OUTPUT_DIR / "Disease_Overlap_Exact_Distribution.csv", index=False
)

fixed_margin_report = pd.DataFrame(
    {
        "Result": [
            "Direct comparisons",
            "Accessions or lines assessed",
            "Observed multidisease count",
            "Exact mean",
            "Median",
            "Central 95% interval",
            "Lower-tail probability",
            "Upper-tail probability",
        ],
        "Value": [
            len(blocks),
            combined_summary["complete_case_n"],
            combined_summary["observed_multidisease_n"],
            f'{combined_summary["exact_null_mean"]:.6f}',
            combined_summary["null_median"],
            f'{combined_summary["null_lower_95"]} to {combined_summary["null_upper_95"]}',
            f'{combined_summary["lower_tail_probability"]:.6f}',
            f'{combined_summary["upper_tail_probability"]:.6f}',
        ],
    }
)
display(fixed_margin_report)


,Result,Value
0,Direct comparisons,5
1,Accessions or lines assessed,556
2,Observed multidisease count,115
3,Exact mean,114.984080
4,Median,115
5,Central 95% interval,105 to 125
6,Lower-tail probability,0.540161
7,Upper-tail probability,0.537208
